# 🌊 Sea Level Monitoring Pipeline
## Copernicus CDS → Parquet → DuckDB

**Project:** Download, explore, and analyze monthly sea level data

**Tech Stack (via `uv`):**
- `earthkit` - Access Copernicus CDS data
- `xarray` & `pandas` - Data processing
- `pyarrow` - Parquet format
- `matplotlib` - Visualization
- `duckdb` - Data warehouse (local, serverless)

## Setup: Using `uv` for Package Management

**Before running this notebook, set up your project with `uv`:**

```bash
mkdir sea-level-monitoring
cd sea-level-monitoring
uv init --python 3.11
uv add earthkit cdsapi xarray pandas pyarrow matplotlib numpy duckdb
uv add --dev jupyter ipython
source .venv/bin/activate
jupyter notebook
```

## 1️⃣ Verify Packages

In [ ]:
packages_to_check = {
    'earthkit': 'earthkit.data',
    'cdsapi': 'cdsapi',
    'xarray': 'xarray',
    'pandas': 'pandas',
    'pyarrow': 'pyarrow',
    'matplotlib': 'matplotlib',
    'duckdb': 'duckdb',
    'numpy': 'numpy'
}

print('🔍 Checking installed packages...\n')

all_installed = True
for package_name, import_name in packages_to_check.items():
    try:
        module = __import__(import_name)
        version = getattr(module, '__version__', 'unknown')
        print(f'✅ {package_name:<15} v{version}')
    except ImportError:
        print(f'❌ {package_name:<15} NOT installed')
        all_installed = False

if all_installed:
    print('\n✅ All packages ready!')
else:
    print('\n⚠️  Run: uv add earthkit cdsapi xarray pandas pyarrow matplotlib numpy duckdb')

## 2️⃣ Import Libraries

In [ ]:
import earthkit.data as ek
import pandas as pd
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
import duckdb
import warnings
from pathlib import Path
from datetime import datetime

# Suppress non-critical warnings for cleaner output
warnings.filterwarnings('ignore')

# Set plotting style - Catppuccin Macchiato theme
plt.style.use('catppuccin-macchiato')

# Create data directory
data_dir = Path('./data')
data_dir.mkdir(exist_ok=True)

print('✅ All libraries imported!')
print(f'📁 Data directory: {data_dir.absolute()}')

### 📝 Understanding `warnings.filterwarnings('ignore')`

**What does it do?**

`warnings.filterwarnings('ignore')` suppresses non-critical warning messages.

**Why?** Libraries emit warnings about deprecated features or future changes. These don't break code—they just clutter output.

**Is it safe?** ✅ Yes! Errors still show up. Only advisory warnings are hidden.

**Other options:**
```python
warnings.filterwarnings('ignore')    # Hide warnings (our choice)
warnings.filterwarnings('default')   # Show warnings (Python default)
warnings.filterwarnings('error')     # Convert warnings to errors
```

## 3️⃣ CDS API Credentials Setup

**Steps:**
1. Go to https://cds.climate.copernicus.eu/
2. Create account & log in
3. Accept terms & conditions
4. Get API key from profile
5. Create `~/.cdsapirc` file:

```
url: https://cds.climate.copernicus.eu/api/v2
key: YOUR_UID:YOUR_API_KEY
```

In [ ]:
cds_config_path = Path.home() / '.cdsapirc'

print('🔐 Checking CDS credentials...\n')

if cds_config_path.exists():
    print('✅ CDS credentials found!')
else:
    print('⚠️  Credentials NOT found')
    print('\nSetup instructions:')
    print('1. Go to https://cds.climate.copernicus.eu/')
    print('2. Log in and accept terms')
    print('3. Go to profile page')
    print('4. Copy UID and API key')
    print('5. Create ~/.cdsapirc with your credentials')
    print('6. chmod 600 ~/.cdsapirc')

## 4️⃣ Download Sea Level Data

**Note:** Uses smart fallback:
- Tries `earthkit.data.retrieve()` first
- Falls back to `cdsapi.Client()` (more reliable)

In [ ]:
dataset_name = 'satellite-sea-level-global'
nc_file = data_dir / 'sea_level_data.nc'

request_params = {
    'product_type': 'monthly_mean_sea_level_anomaly',
    'variable': 'sea_level_anomaly',
    'year': ['2019', '2020', '2021', '2022', '2023', '2024'],
    'month': ['01', '02', '03', '04', '05', '06', '07', '08', '09', '10', '11', '12'],
    'format': 'netcdf'
}

print('📥 Downloading sea level data...\n')
print(f'Dataset: {dataset_name}')
print(f'Period: 2019-2024')
print(f'Output: {nc_file}')
print('\n⏳ This may take 2-5 minutes...\n')

try:
    # Try earthkit first
    print('Trying earthkit.data.retrieve()...')
    data = ek.data.retrieve(
        dataset_name,
        request_params
    )
    data.save(str(nc_file))
    print('✅ Download complete (earthkit)!')
    
except AttributeError:
    # Fallback to cdsapi
    print('Using cdsapi.Client()...\n')
    import cdsapi
    
    client = cdsapi.Client()
    client.retrieve(
        dataset_name,
        request_params,
        str(nc_file)
    )
    print('✅ Download complete (cdsapi)!')
    
except Exception as e:
    print(f'❌ Download failed: {type(e).__name__}')
    print(f'Error: {e}')
    print('\nTroubleshooting:')
    print('- Check ~/.cdsapirc exists and has correct format')
    print('- Run: chmod 600 ~/.cdsapirc')
    print('- Check internet connection')
    print('- Try smaller date range first')

if nc_file.exists():
    size_mb = nc_file.stat().st_size / 1024 / 1024
    print(f'\n📁 File size: {size_mb:.2f} MB')

## 5️⃣ Load & Explore Data

In [ ]:
nc_file = data_dir / 'sea_level_data.nc'

if nc_file.exists():
    print('🔓 Loading NetCDF file...\n')
    ds = xr.open_dataset(nc_file)
    
    print('📊 Dataset:')
    print(ds)
    print('\n' + '='*70)
    print('📋 Summary:')
    print(f'  Dimensions: {dict(ds.dims)}')
    print(f'  Variables: {list(ds.data_vars)}')
    print(f'  Coordinates: {list(ds.coords)}')
else:
    print(f'❌ File not found: {nc_file}')
    print('Complete download step above first')

## 6️⃣ Data Quality Check

In [ ]:
print('🔍 Data Quality Check:\n')

for var in ds.data_vars:
    total = ds[var].size
    missing = int(ds[var].isnull().sum().values)
    pct = (missing / total) * 100 if total > 0 else 0
    
    print(f'{var}:')
    print(f'  Total: {total:,}')
    print(f'  Missing: {missing:,} ({pct:.1f}%)')
    print(f'  Type: {ds[var].dtype}\n')

print('='*70)
print('📊 Statistics:\n')
for var in ds.data_vars:
    print(f'{var}:')
    print(ds[var].describe())

## 7️⃣ Visualization - Maps & Time Series

In [ ]:
# Global sea level anomaly map (latest month)
if 'sla' in ds.data_vars:
    print('📈 Generating global map...\n')
    
    latest_sla = ds['sla'].isel(time=-1)
    
    fig, ax = plt.subplots(figsize=(14, 8))
    latest_sla.plot(
        ax=ax,
        cmap='RdBu_r',
        cbar_kwargs={'label': 'Sea Level Anomaly (m)'},
        robust=True
    )
    
    time_str = pd.Timestamp(latest_sla.time.values).strftime('%Y-%m')
    ax.set_title(f'Global Sea Level Anomaly - {time_str}', fontsize=14, fontweight='bold')
    ax.set_xlabel('Longitude (°)')
    ax.set_ylabel('Latitude (°)')
    
    plt.tight_layout()
    map_file = data_dir / 'sea_level_map.png'
    plt.savefig(map_file, dpi=300, bbox_inches='tight')
    plt.show()
    
    print(f'✅ Saved: {map_file}')

In [ ]:
# Time series: Global mean sea level
if 'sla' in ds.data_vars:
    print('📈 Generating time series...\n')
    
    global_mean = ds['sla'].mean(dim=['longitude', 'latitude'])
    
    fig, ax = plt.subplots(figsize=(13, 6))
    global_mean.plot(
        ax=ax,
        linewidth=2.5,
        marker='o',
        markersize=5,
        color='steelblue'
    )
    
    # Add trend line
    z = np.polyfit(range(len(global_mean)), global_mean.values, 1)
    p = np.poly1d(z)
    ax.plot(range(len(global_mean)), p(range(len(global_mean))), 
            'r--', linewidth=2, label='Trend')
    
    ax.set_title('Global Mean Sea Level Anomaly', fontsize=14, fontweight='bold')
    ax.set_xlabel('Date')
    ax.set_ylabel('Sea Level Anomaly (m)')
    ax.grid(True, alpha=0.3)
    ax.legend()
    
    plt.tight_layout()
    ts_file = data_dir / 'sea_level_timeseries.png'
    plt.savefig(ts_file, dpi=300, bbox_inches='tight')
    plt.show()
    
    print(f'✅ Saved: {ts_file}')

## 8️⃣ Transform to Parquet

In [ ]:
print('🔄 Converting NetCDF → Parquet...\n')

if 'sla' in ds.data_vars:
    # Convert to DataFrame
    sla_data = ds['sla'].to_pandas()
    df = sla_data.reset_index()
    
    # Melt to long format
    df_melted = df.melt(
        id_vars=['time'],
        var_name='location',
        value_name='sea_level_anomaly_m'
    )
    
    # Extract lat/lon
    df_melted['latitude'] = df_melted['location'].apply(
        lambda x: float(str(x).split(',')[0].replace('(', ''))
    )
    df_melted['longitude'] = df_melted['location'].apply(
        lambda x: float(str(x).split(',')[1].replace(')', ''))
    )
    
    df_melted = df_melted.drop('location', axis=1)
    df_melted['timestamp'] = pd.to_datetime(df_melted['time'])
    df_melted['year'] = df_melted['timestamp'].dt.year
    df_melted['month'] = df_melted['timestamp'].dt.month
    df_melted['date'] = df_melted['timestamp'].dt.date
    
    # Clean NaNs
    df_melted = df_melted.dropna(subset=['sea_level_anomaly_m'])
    
    # Reorder columns
    df_melted = df_melted[[
        'timestamp', 'date', 'year', 'month',
        'latitude', 'longitude',
        'sea_level_anomaly_m'
    ]]
    
    print(f'📊 DataFrame:')
    print(f'  Shape: {df_melted.shape}')
    print(f'  Rows: {len(df_melted):,}')
    print(f'  Columns: {list(df_melted.columns)}')
    print(f'\n🔍 Sample:')
    print(df_melted.head(10))
    
    # Save to Parquet
    parquet_file = data_dir / 'sea_level_data.parquet'
    df_melted.to_parquet(
        parquet_file,
        compression='snappy',
        index=False,
        engine='pyarrow'
    )
    
    print(f'\n✅ Parquet created!')
    print(f'  Path: {parquet_file}')
    print(f'  Size: {parquet_file.stat().st_size / 1024 / 1024:.2f} MB')

## 9️⃣ Load into DuckDB

In [ ]:
parquet_file = data_dir / 'sea_level_data.parquet'
db_file = data_dir / 'sea_level_monitoring.duckdb'

print('🗄️  Initializing DuckDB...\n')

conn = duckdb.connect(str(db_file))
print(f'✅ Connected to {db_file}')

# Create table from parquet
try:
    conn.execute(
        f"CREATE TABLE sea_level_monthly AS SELECT * FROM read_parquet('{parquet_file}')"
    )
    print('✅ Table created: sea_level_monthly')
except:
    print('⚠️  Table exists, dropping and recreating...')
    conn.execute('DROP TABLE IF EXISTS sea_level_monthly')
    conn.execute(
        f"CREATE TABLE sea_level_monthly AS SELECT * FROM read_parquet('{parquet_file}')"
    )
    print('✅ Table recreated')

# Show table info
result = conn.execute('SELECT COUNT(*) FROM sea_level_monthly').fetchall()
print(f'\n  Rows: {result[0][0]:,}')
print('\n📋 Schema:')
schema = conn.execute('DESCRIBE sea_level_monthly').fetchall()
for col_name, col_type, *_ in schema:
    print(f'  • {col_name}: {col_type}')

## 🔟 SQL Queries

In [ ]:
print('📊 Query 1: Mean by Year\n')

query1 = '''
SELECT
    year,
    COUNT(*) as measurements,
    AVG(sea_level_anomaly_m) as mean_m,
    MIN(sea_level_anomaly_m) as min_m,
    MAX(sea_level_anomaly_m) as max_m
FROM sea_level_monthly
GROUP BY year
ORDER BY year
'''

df_result = conn.execute(query1).df()
print(df_result.to_string(index=False))

In [ ]:
print('\n📊 Query 2: Extreme Locations (Latest Month)\n')

query2 = '''
SELECT
    latitude,
    longitude,
    sea_level_anomaly_m,
    timestamp
FROM sea_level_monthly
WHERE timestamp = (SELECT MAX(timestamp) FROM sea_level_monthly)
ORDER BY sea_level_anomaly_m DESC
LIMIT 5
'''

df_result = conn.execute(query2).df()
print('Highest anomalies:')
print(df_result.to_string(index=False))

## Summary

✅ **Pipeline Complete!**

```
Copernicus CDS → NetCDF → Parquet → DuckDB → SQL Analysis
```

**Next Steps (Phase 2):**
- Seasonal analysis
- Feature engineering
- Time series forecasting
- Machine learning models